# Lab 3 - Advanced Queries

**Objective:** write production-grade analytical SQL (window functions, CTEs, joins) and reason about their performance with `EXPLAIN ANALYZE`. End with an Iceberg time-travel example.

**Estimated time:** ~25 minutes.

**You will:**

1. Rank airports per continent using `ROW_NUMBER()`.
2. Compute a 3-month rolling average with a windowed `AVG(...) OVER (...)`.
3. Join two aggregates with a CTE to compute per-pilot delay rates.
4. Use `SHOW STATS` and `EXPLAIN ANALYZE` to tune queries.
5. Query an older Iceberg snapshot with time-travel syntax.

**Prerequisites:** Lab 1 completed successfully.

In [ ]:
import os
import pandas as pd
from trino.dbapi import connect
from trino.auth import BasicAuthentication

conn = connect(
    host=os.environ['TRINO_HOST'],
    port=int(os.environ.get('TRINO_PORT', 443)),
    user=os.environ['TRINO_USER'],
    auth=BasicAuthentication(os.environ['TRINO_USER'], os.environ['TRINO_PASSWORD']),
    http_scheme='https',
    catalog='iceberg',
    schema='airline_lab',
)

USERNAME = os.environ['TRINO_USER']

def q(sql):
    cur = conn.cursor(); cur.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

## 1 - `ROW_NUMBER`: top-5 airports per continent

Demonstrates the canonical *rank within partition* pattern. The subquery computes the rank; the outer query keeps only the top 5 per partition.

In [ ]:
q(f'''
    SELECT *
    FROM (
        SELECT airport_continent,
               airport_name,
               COUNT(*) AS flights,
               ROW_NUMBER() OVER (PARTITION BY airport_continent ORDER BY COUNT(*) DESC) AS rnk
        FROM iceberg.airline_lab.flights_{USERNAME}
        GROUP BY airport_continent, airport_name
    )
    WHERE rnk <= 5
    ORDER BY airport_continent, rnk
''')

## 2 - CTE + rolling window: 3-month moving average

A `WITH` clause makes the intermediate monthly count readable. The outer `AVG(...) OVER (...)` produces a smoothed trend that dampens noise in short periods.

In [ ]:
q(f'''
    WITH monthly AS (
        SELECT date_trunc('month', departure_date) AS month,
               COUNT(*) AS flights
        FROM iceberg.airline_lab.flights_{USERNAME}
        GROUP BY date_trunc('month', departure_date)
    )
    SELECT month,
           flights,
           AVG(flights) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS rolling_avg_3m
    FROM monthly
    ORDER BY month
''')

## 3 - Join: delay rate per pilot

Combines two aggregates (`per_pilot` and `delayed`) with a `LEFT JOIN`. The optimizer decides whether to broadcast the smaller side - you can influence that decision with `SET SESSION join_distribution_type = 'BROADCAST'` (use sparingly).

In [ ]:
q(f'''
    WITH per_pilot AS (
        SELECT pilot_name, COUNT(*) AS total_flights
        FROM iceberg.airline_lab.flights_{USERNAME}
        GROUP BY pilot_name
    ),
    delayed AS (
        SELECT pilot_name, COUNT(*) AS delays
        FROM iceberg.airline_lab.flights_{USERNAME}
        WHERE flight_status = 'Delayed'
        GROUP BY pilot_name
    )
    SELECT p.pilot_name,
           p.total_flights,
           COALESCE(d.delays, 0) AS delays,
           ROUND(100.0 * COALESCE(d.delays, 0) / p.total_flights, 2) AS delay_pct
    FROM per_pilot p
    LEFT JOIN delayed d ON p.pilot_name = d.pilot_name
    ORDER BY delay_pct DESC
    LIMIT 20
''')

## 4 - Performance tuning

### `SHOW STATS FOR` - feed the optimizer

Trino's cost-based optimizer needs row counts and null fractions. Run `ANALYZE TABLE ...` after any large load, then verify the stats are populated.

In [ ]:
q(f'SHOW STATS FOR iceberg.airline_lab.flights_{USERNAME}')

### `EXPLAIN ANALYZE` - measure what actually ran

Unlike `EXPLAIN`, this executes the query and reports per-stage wall time, CPU time, and actual row counts. Look for stages where *estimated* and *actual* row counts diverge - that usually means stale statistics.

In [ ]:
cur = conn.cursor()
cur.execute(f'''
    EXPLAIN ANALYZE
    SELECT airport_continent, COUNT(*)
    FROM iceberg.airline_lab.flights_{USERNAME}
    WHERE departure_date BETWEEN DATE '2022-01-01' AND DATE '2022-06-30'
    GROUP BY airport_continent
''')
print(cur.fetchone()[0])

### Performance tips

* Prefer `APPROX_DISTINCT()` over `COUNT(DISTINCT ...)` when exact cardinality is not required. It is a single-pass HyperLogLog implementation.
* Partition Iceberg tables with transforms (`year(...)`, `bucket(n, col)`) rather than raw columns. This avoids small-file explosions as partition keys change.
* Leave `join_distribution_type` at the default `AUTOMATIC` unless `EXPLAIN ANALYZE` shows the optimizer picking the wrong strategy.
* Push predicates down. Iceberg prunes entire partitions and even individual files using column min/max statistics.
* Always `ANALYZE TABLE` after bulk writes so the optimizer has accurate row counts.